# 01 — The Cast: Driver, Executor, Task, Worker

**Target:** Apache Spark 4.2.0; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Does every PySpark DataFrame operation create a Python worker?

**Why this matters:** Python is often the API used to build a plan, while Spark SQL may execute built-in expressions in the JVM. Python workers are launched for Python execution paths, not for the mere fact that the caller imported `pyspark`.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import os, sys, time

os.environ['PYSPARK_PYTHON'] = sys.executable
spark = (SparkSession.builder.master('local[2]').appName('thread-lab-01')
         .config('spark.ui.enabled', 'true').getOrCreate())
print('Spark:', spark.version, 'Python:', sys.version.split()[0], 'PID:', os.getpid())
print('Spark UI:', spark.sparkContext.uiWebUrl or 'disabled')

In [ ]:
base = spark.range(0, 100_000).withColumn('bucket', F.col('id') % 7)
print('Built-in SQL result:', base.groupBy('bucket').count().collect())

In [ ]:
@F.udf(returnType=T.LongType())
def python_identity(value):
    return value

print('Python UDF result:', base.select(python_identity('id').alias('id')).limit(5).collect())
time.sleep(2) # leave a moment to inspect the process tree and Spark UI

## Observe
Open the Spark UI URL printed below while the session is active (`4040` is the default; Spark increments the port if it is busy). Compare the SQL-only and Python-UDF stages. On Linux/macOS, inspect child processes with `ps -ef --forest` or `pstree -ap <driver-pid>`; on Windows, use Task Manager or Process Explorer. Optional: install `psutil` to inspect children from Python.

**Expected:** the built-in query runs in the JVM. The UDF stage can start a Python worker. In `local[2]`, the driver and local executor backend usually share one Spark JVM; you should not expect a separate executor JVM. On a cluster, executors run in their own JVM processes. Worker reuse affects what remains visible afterward.

**Why:** a Python driver and Python worker are different processes with different roles. Spark Connect also changes the client/server topology. Never infer worker count from the API language alone.

**Cleanup:** `spark.stop()` when finished.

In [ ]:
spark.stop()